# A1.7 · Model routing architecture

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Both directions*

---

**Risk.** The router fails open to a weaker model under load — a silent downgrade of every guardrail.

**Control.** Treat tier selection as a security decision with explicit fail-closed defaults.

**This lab.** Prove the router fails closed, not open.

| | |
|---|---|
| Open-source tooling | LiteLLM, vLLM |
| Open-weight models | GLM-4.6, Llama 3.3, Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A1.7"))

Routing between models is a security decision, not a cost decision — because the cheap model is usually the one holding the tools.

In [ ]:
from cybercommons import planes

ROUTES = {
 # (model, what it is allowed to trigger)
 "small local (Llama 3.3 8B)": planes.Manifest("router:small",
     [planes.Tool("read_file"), planes.Tool("search_code")], rung="L1"),
 "mid open-weight (GLM-4.6)": planes.Manifest("router:mid",
     [planes.Tool("read_file"),
      planes.Tool("write_file", writes=True, scope="project")],
     approval_required={"write_file"}, rung="L2"),
 "large (Kimi K2) for planning only": planes.Manifest("router:large",
     [planes.Tool("read_file")], rung="L1"),
}
for name, m in ROUTES.items():
    print(f"{name:36s} blast={m.blast_radius()['total']:3d}  "
          f"issues={m.rung_check() or 'none'}")

print("\nThe routing rule that matters:")
print("  capability decides which model *plans*;")
print("  blast radius decides which model is allowed to *act*.")

The common anti-pattern inverts this: the expensive model plans, and the cheap fast model is given the tools so the loop stays responsive. That puts the weakest reasoning next to the highest authority.

### Expect

All three routes report a blast radius of 0 or near it and no rung problems, because the tools are attached to the gated route rather than the fastest one.

### Your turn

Model the anti-pattern: give `router:small` the ungated `write_file` and `deploy_prod`. Compare the blast radius, then argue the cost saving against it.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A1.7.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*